Датасет был загружен и были показаны первые 5 строк

In [1]:
import pandas as pd 
df=pd.read_csv('../data/ml-1m/ratings.dat',sep='::',names=['user_id','movie_id','rating','timestamp'],engine='python')
df.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


Далее была проверена размерность датасета

In [2]:
print(f'Количество строк в датасете {df.shape[0]}, количество признаков - {df.shape[1]}')

Количество строк в датасете 1000209, количество признаков - 4


Далее выводится информация по каждой колонке

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1000209 non-null  int64
 1   movie_id   1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


Пропущенные значения в датасете отсутствуют. Все столбцы представлены в числовом формате. Датасет содержит информацию об оценках фильмов пользователями:
- user_id — уникальный идентификатор пользователя
- movie_id — уникальный идентификатор фильма
- rating — оценка, выставленная пользователем фильму
- timestamp — время выставления оценки в формате Unix timestamp

Далее провяется количество пропусков методом для точной проверки


In [4]:
df.isna().sum()

user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64

Подтвердилось - пропуски отсутствуют, далее будут проверены дубликаты

In [5]:
df.duplicated().sum()

np.int64(0)

Дубликаты аналогично отсутствуют. Теперь необходимо проверить состав класса рейтинга

In [6]:
df.rating.value_counts().sort_index()

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

Сейчас класс рейтинга находится в явном дисбалансе, но в таком виде он и не будет использоваться далее. Была введена новая колонка `interaction`
Правило:
Если оценка >=4, взаимодействие положительное (1)
Остальные ситуации для матрицы нас не интересуют


In [7]:
df = df[df["rating"] >= 4].copy()
df["interaction"] = 1

In [8]:
df.head(5)

,user_id,movie_id,rating,timestamp,interaction
0,1,1193,5,978300760,1
3,1,3408,4,978300275,1
4,1,2355,5,978824291,1
6,1,1287,5,978302039,1
7,1,2804,5,978300719,1


In [9]:
print("Пользователей:", df["user_id"].nunique())
print("Фильмов:", df["movie_id"].nunique())
print("Положительных взаимодействий:", len(df))

Пользователей: 6038
Фильмов: 3533
Положительных взаимодействий: 575281


Далее необходимо привести timestamp к нормальной дате

In [10]:
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s")

df[["timestamp", "datetime"]].head()

,timestamp,datetime
0,978300760,2000-12-31 22:12:40
3,978300275,2000-12-31 22:04:35
4,978824291,2001-01-06 23:38:11
6,978302039,2000-12-31 22:33:59
7,978300719,2000-12-31 22:11:59


Далее были просмотрены диапазоны дат

In [11]:
print("Первая оценка:", df["datetime"].min())
print("Последняя оценка:", df["datetime"].max())

Первая оценка: 2000-04-25 23:05:32
Последняя оценка: 2003-02-28 17:49:50


In [12]:
user_interactions = df.groupby("user_id").size()

user_interactions.describe()

count    6038.000000
mean       95.276747
std       105.005005
min         1.000000
25%        27.000000
50%        58.000000
75%       124.000000
max      1435.000000
dtype: float64

In [13]:
print("Пользователей с одним положительным взаимодействием:",
      (user_interactions == 1).sum())

Пользователей с одним положительным взаимодействием: 1


In [14]:
user_interactions = df.groupby("user_id").size()

valid_users = user_interactions[user_interactions >= 2].index

df = df[df["user_id"].isin(valid_users)].copy()

In [15]:
df.groupby("user_id").size().min()

np.int64(2)

Данные были разделены на обучающую и валидационную выборки по времени. Для каждого пользователя взаимодействия с максимальным timestamp были отнесены к validation, а все более ранние взаимодействия — к train. Пользователи, имеющие взаимодействия только в один момент времени, были исключены, так как для них невозможно сформировать временное разделение без утечки информации. Дополнительно была выполнена проверка, что для каждого пользователя максимальное время взаимодействия в train строго меньше минимального времени взаимодействия в validation

In [16]:
user_timestamps = df.groupby("user_id")["timestamp"].nunique()

valid_users = user_timestamps[user_timestamps >= 2].index
df = df[df["user_id"].isin(valid_users)].copy()

max_timestamp = df.groupby("user_id")["timestamp"].transform("max")

val_df = df[df["timestamp"] == max_timestamp].copy()
train_df = df[df["timestamp"] < max_timestamp].copy()

train_last = train_df.groupby("user_id")["timestamp"].max()
val_first = val_df.groupby("user_id")["timestamp"].min()

print("Корректность временного разделения:",
      (train_last < val_first).all())

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print("Пользователей в train:", train_df["user_id"].nunique())
print("Пользователей в validation:", val_df["user_id"].nunique())

Корректность временного разделения: True
Train: (565861, 6)
Validation: (9417, 6)
Пользователей в train: 6036
Пользователей в validation: 6036


Считаем популярность фильмов только по train

In [17]:
popularity = (
    train_df
    .groupby("movie_id")
    .size()
    .sort_values(ascending=False)
)
popularity.head(10)

movie_id
2858    2817
260     2582
1196    2486
1198    2242
2028    2241
593     2226
2571    2140
2762    2123
1210    2111
527     2050
dtype: int64

In [18]:
popular_items = popularity.index.tolist()

Посмотрим историю одного пользователя

In [19]:
user_id = 1

seen_items = set(
    train_df.loc[
        train_df["user_id"] == user_id,
        "movie_id"
    ]
)

seen_items

{1,
 150,
 260,
 527,
 531,
 588,
 594,
 595,
 608,
 783,
 919,
 938,
 1022,
 1028,
 1029,
 1035,
 1097,
 1193,
 1207,
 1246,
 1270,
 1287,
 1545,
 1566,
 1721,
 1836,
 1907,
 1961,
 1962,
 2018,
 2028,
 2294,
 2355,
 2398,
 2692,
 2762,
 2791,
 2797,
 2804,
 2918,
 3105,
 3114,
 3186,
 3408}

Рекомендуем популярные фильмы, которых пользователь ещё не видел

In [20]:
recommendations = [
    movie_id
    for movie_id in popular_items
    if movie_id not in seen_items
][:10]

recommendations

[2858, 1196, 1198, 593, 2571, 1210, 318, 589, 858, 110]

In [21]:
actual_items = set(
    val_df.loc[
        val_df["user_id"] == user_id,
        "movie_id"
    ]
)

actual_items

{48}

In [22]:
def recommend_popular(user_id, train_df, popular_items, k=10):
    seen_items = set(
        train_df.loc[
            train_df["user_id"] == user_id,
            "movie_id"
        ]
    )

    recommendations = [
        movie_id
        for movie_id in popular_items
        if movie_id not in seen_items
    ]

    return recommendations[:k]

In [23]:
recommend_popular(
    user_id=1,
    train_df=train_df,
    popular_items=popular_items,
    k=10
)

[2858, 1196, 1198, 593, 2571, 1210, 318, 589, 858, 110]

In [24]:
recommend_popular(1, train_df, popular_items, 10)

[2858, 1196, 1198, 593, 2571, 1210, 318, 589, 858, 110]

In [25]:
relevance = [
    1 if movie_id in actual_items else 0
    for movie_id in recommendations
]

relevance

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [26]:
import math
ideal_relevance = sorted(relevance, reverse=True)

idcg = 0

for i, rel in enumerate(ideal_relevance):
    idcg += rel / math.log2(i + 2)

ndcg = dcg / idcg if idcg > 0 else 0

ndcg

0

In [27]:
def ndcg_at_k(recommendations, actual_items, k=10):
    relevance = [
        1 if movie_id in actual_items else 0
        for movie_id in recommendations[:k]
    ]

    dcg = sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevance)
    )

    ideal_hits = min(len(actual_items), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0

In [28]:
ndcg_scores = []

for user_id in val_df["user_id"].unique():

    recommendations = recommend_popular(
        user_id=user_id,
        train_df=train_df,
        popular_items=popular_items,
        k=10
    )

    actual_items = set(
        val_df.loc[
            val_df["user_id"] == user_id,
            "movie_id"
        ]
    )

    score = ndcg_at_k(
        recommendations=recommendations,
        actual_items=actual_items,
        k=10
    )

    ndcg_scores.append(score)

In [29]:
import numpy as np

mean_ndcg = np.mean(ndcg_scores)

print("Mean NDCG@10:", mean_ndcg)

Mean NDCG@10: 0.02309822430487727


In [30]:
hits = sum(score > 0 for score in ndcg_scores)

print("Пользователей с хотя бы одним попаданием:", hits)
print("Всего пользователей:", len(ndcg_scores))
print("Доля пользователей с попаданием:", hits / len(ndcg_scores))

Пользователей с хотя бы одним попаданием: 369
Всего пользователей: 6036
Доля пользователей с попаданием: 0.06113320079522863


Зафиксируем baseline

In [ ]:
baseline_results={
    "model":"popularity",
    "k":10,
    "ndcg":mean_ndcg,
    "hit_rate": hits / len(ndcg_scores)
}
baseline_results

SyntaxError: invalid syntax (116953633.py, line 2)